# Coach-change prediction

Compare a naive model, an SVM, and a KNN. Each model predicts `coach_changed` using the three previous seasons.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
TARGET = "coach_changed"
TEST_START_YEAR = 2005
HISTORY_YEARS = 3

# Each model stores only its final test contingency table.
model_confusion_matrices = {}

teams = pd.read_csv(Path("data_prepared") / "teams.csv")
df = teams.sort_values(["franchID", "year"]).copy()

numeric_columns = [
    "rank", "conference_rank", "prev_conference_rank", TARGET,
]
categorical_columns = []

for column in numeric_columns:
    for lag in range(1, HISTORY_YEARS + 1):
        df[f"previous_{lag}_{column}"] = df.groupby("franchID")[column].shift(lag)

sequence_columns = [
    column for column in numeric_columns
    if column != TARGET and df.loc[df["year"] <= TEST_START_YEAR - 4, column].notna().any()
]
previous_numeric = [
    f"previous_{lag}_{column}"
    for lag in range(1, HISTORY_YEARS + 1)
    for column in numeric_columns
    if df.loc[df["year"] <= TEST_START_YEAR - 4, f"previous_{lag}_{column}"].notna().any()
]
features = previous_numeric + categorical_columns
df = df[df["year"] >= df["year"].min() + HISTORY_YEARS].copy()
train = df[df["year"] < TEST_START_YEAR].copy()
test = df[df["year"] >= TEST_START_YEAR].copy()

def store_model_contingency_table(name, predictions):
    model_confusion_matrices[name] = confusion_matrix(
        test[TARGET],
        predictions,
        labels=[0, 1],
    )

naive_predictions = np.zeros(len(test), dtype=int)
store_model_contingency_table("Naive model", naive_predictions)

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, previous_numeric),
    ("categorical", categorical_pipeline, categorical_columns),
])


In [ ]:
svm = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            SVC(
                kernel="poly",
                degree=3,
                C=0.1,
                gamma=0.01,
                class_weight={0: 1, 1: 15},
                probability=True,
                random_state=RANDOM_STATE,
                tol=1e-3,
            ),
        ),
    ]
)

fit = train[train["year"] <= train["year"].max() - 3]
validation = train[train["year"] > train["year"].max() - 3]
svm.fit(fit[features], fit[TARGET])
validation_probability = svm.predict_proba(validation[features])[:, 1]
thresholds = np.arange(0.20, 0.81, 0.05)
threshold = float(
    thresholds[
        np.argmax(
            [
                f1_score(
                    validation[TARGET], validation_probability >= value, zero_division=0
                )
                for value in thresholds
            ]
        )
    ]
)
svm.fit(train[features], train[TARGET])
svm_predictions = (svm.predict_proba(test[features])[:, 1] >= threshold).astype(int)
store_model_contingency_table(
    f"SVM (threshold={threshold:.2f})",
    svm_predictions,
)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(
        n_neighbors=1,
        weights="distance",
        metric="euclidean",
    )),
])

# Train on the validation-training period
knn.fit(fit[features], fit[TARGET])

validation_probability = knn.predict_proba(
    validation[features]
)[:, 1]

# Select the threshold using validation data only
thresholds = np.arange(0.05, 0.81, 0.05)

validation_f1 = [
    f1_score(
        validation[TARGET],
        validation_probability >= threshold,
        zero_division=0,
    )
    for threshold in thresholds
]

knn_threshold = float(
    thresholds[int(np.argmax(validation_f1))]
)

print(f"Best validation F1: {max(validation_f1):.3f}")
print(f"Selected threshold: {knn_threshold:.2f}")

# Refit on all training seasons
knn.fit(train[features], train[TARGET])

test_probability = knn.predict_proba(
    test[features]
)[:, 1]

knn_predictions = (
    test_probability >= knn_threshold
).astype(int)

store_model_contingency_table(
    f"KNN (k=5, threshold={knn_threshold:.2f})",
    knn_predictions,
)



In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import seaborn as sns

def scores_from_contingency_table(cm):
    true_negative, false_positive, false_negative, true_positive = cm.ravel()
    total = cm.sum()
    accuracy = (true_positive + true_negative) / total
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    recall_0 = true_negative / (true_negative + false_positive) if true_negative + false_positive else 0.0
    balanced_accuracy = (recall_0 + recall) / 2
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }

model_scores = {
    name: scores_from_contingency_table(cm)
    for name, cm in model_confusion_matrices.items()
}
scores_table = pd.DataFrame.from_dict(model_scores, orient="index")

print("\nContingency tables:")
for name, cm in model_confusion_matrices.items():
    print(f"\n{name}")
    print(cm)

print("\nModel scores:")
print(scores_table.round(3))

scores_for_plot = scores_table.reset_index().rename(columns={"index": "model"}).melt(
    id_vars="model",
    var_name="score_type",
    value_name="score",
)
plt.figure(figsize=(11, 5))
sns.barplot(
    data=scores_for_plot,
    x="score_type",
    y="score",
    hue="model",
)
plt.ylim(0, 1)
plt.xlabel("Score type")
plt.ylabel("Score")
plt.title("Model comparison on the test set")
plt.xticks(rotation=20)
plt.legend(title="Model")
plt.tight_layout()
plt.show()
